# REVE v5: v4 + аугментации против амплитуды и наклона спектра (общий набор Г1)

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-27 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Разбор ошибок REVE v4 (`reports/2026-10-27_reve_errors.md`): оценка растёт с амплитудой записи внутри классов (Спирмен +0,34…+0,57), хотя каждое окно нормируется. Вероятный путь — форма спектра (в высокоамплитудных записях больше медленных волн). Здесь при обучении случайно меняются наклон спектра, мощность 1–4 Гц и усиление каналов. Главная проверка — упала ли связь оценки с амплитудой внутри классов при том же ROC AUC |
| **Данные** | как v4: общий набор `zakharsann/h1-prep-shared` через `shared_data.py`, ФВЧ 0,3 Гц по записи, деление NMT train на fit / val (отложенные 15 %) — столбец `split` |
| **Обучение** | всё как v4 (новое случайное окно на каждой эпохе, `brain-bzh/reve-base` + голова, основа 1·10⁻⁵, голова 5·10⁻⁴, до 10 эпох, остановка по отложенным 15 %, 3 запуска seed 0/1/2) + **новые аугментации до нормировки окна** (ниже) |
| **Оценка** | как v4: оценки всех окон; основные цифры — 60 окон, среднее верхних 20 % (и медиана); ансамбль 3 запусков; порог Юдена и чувствительности 80 % по отложенным 15 %, перенос без изменений. Спирмен оценка–амплитуда (`std_uv`) внутри классов. Сравнение с v4, калибровка на 20 турецких людях и ансамбль с EEG-DINO — по csv (раздел 5) |
| **Бюджет** | ≤ 100 мин T4 (замер скорости не повторяем — модель та же) |

**Почему такие аугментации.**
- **Общий масштаб амплитуды записи/окна — не берём.** Нормировка окна (вычесть среднее канала, поделить на общее СКО окна, обрезка ±15) убирает любой общий множитель полностью: `rv_input(c·W) = rv_input(W)`. Такая аугментация ничего бы не меняла.
- **Случайное усиление каналов** (p = 0,5, множитель на канал log-равномерно 0,8–1,25) — нормировка его не убирает: делится только общее СКО, соотношение каналов остаётся. Ломает опору на то, какие каналы «громче».
- **Наклон спектра** (p = 0,8): спектр окна умножается на (f / 10 Гц)^α, α ~ U(−0,5; 0,5), f ограничена снизу 0,5 Гц. При α < 0 медленные волны усиливаются (как в высокоамплитудной записи), при α > 0 — ослабляются. Модель видит одно и то же окно с разной долей медленных волн при той же метке — опора на «общий наклон спектра» становится невыгодной.
- **Полоса 1–4 Гц** (p = 0,5): мощность дельта-полосы × U(0,7; 1,4) по амплитуде (≈ ±3 дБ), края сглажены (0,5 Гц). Слабая сила, чтобы не стирать очаговые медленные волны — они могут быть настоящим признаком.
- Наклон и дельта общие для всех каналов окна (это свойство записи, а не канала). Делается через FFT на видеокарте; перед FFT окно дополняется зеркально на 1 с с каждой стороны, чтобы не было скачка на стыке краёв. Затем нормировка окна и прежние аугментации v4 (шум, зануление каналов, замена полушарий, сдвиг).

**Веса в выходе блокнота** (в git не кладутся): `reve_v5_seed{0,1,2}.pt` — тот же формат, что у v4 (`{"state_dict", "config"}`, класс `RVNet` и `rv_input` в `reve_v5_model.py`). Подготовка входа при применении та же, аугментации — только при обучении.


In [ ]:
MODEL_SRC = r'''"""REVE v5 (Г1): модель и подготовка окна. Веса — reve_v5_seed{0,1,2}.pt выхода блокнота zakharsann/h1-reve-v5-aug.

    import torch
    from reve_v5_model import RVNet, rv_input, positions
    ck = torch.load("reve_v5_seed0.pt", map_location="cpu")
    m = RVNet(positions()); m.load_state_dict(ck["state_dict"]); m.eval()
    p = torch.sigmoid(m(rv_input(W)))      # W: (окна, 19, 2000), мкВ, 200 Гц, каналы в порядке EL
"""
import numpy as np, torch, torch.nn as nn
from transformers import AutoModel

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
NEW = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}


def positions(device="cpu"):
    """Координаты электродов для REVE: (19, 3)."""
    bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
    try:
        pos = bank([NEW.get(e, e) for e in EL])
    except Exception:
        pos = bank(EL)
    return pos.to(device)


def rv_input(W):
    """Окно -> из каждого канала вычесть среднее, всё окно поделить на общее СКО по всем каналам и отсчётам, обрезка ±15."""
    x = np.asarray(W, np.float32)
    x = x - x.mean(-1, keepdims=True)
    x = x / (x.std(axis=(1, 2), keepdims=True) + 1e-6)
    return torch.from_numpy(np.clip(x, -15, 15))


class RVNet(nn.Module):
    """REVE (22 слоя) + голова: среднее по времени с сохранением каналов -> 19*512 -> два линейных слоя."""
    def __init__(self, pos, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(19 * 512, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.register_buffer("pos", pos, persistent=False)

    def forward(self, x):
        o = self.backbone(x, self.pos.expand(x.size(0), -1, -1))
        if isinstance(o, (tuple, list)): o = o[0]
        if hasattr(o, "last_hidden_state"): o = o.last_hidden_state
        assert o.dim() == 4 and o.shape[1] == 19, tuple(o.shape)
        return self.head(o.float().mean(2).flatten(1)).squeeze(-1)
'''

In [ ]:
"""Общий набор Г1 (NMT + турецкий) — выход блокнота Kaggle zakharsann/h1-prep-shared.

Подключение в kernel-metadata.json: "kernel_sources": ["zakharsann/h1-prep-shared"].
Блокноты моделей копируют этот файл в начало (одна ячейка) и пользуются функциями ниже.

Формат (подробно — в описании блокнота h1_2026-10-26_zs_prep-shared.ipynb):
- h1_shared_meta.csv — по строке на запись: dataset, part, split, id, y, ..., shard200/off200/n200 (и shard256/... если есть);
- h1_shared_info.json — фильтр, множитель масштаба NMT, сколько минут NMT train сохранено, порядок каналов;
- fs200/<имя>.npy — плоские массивы float16: записи (19, n) в C-порядке подряд, запись = arr[off : off + 19*n].
Сигнал в мкВ, 19 каналов в порядке EL, средний референт, 0,1-75 Гц + режекция 50 Гц. Окна НЕ нормированы.
"""
import glob, json, os
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, scipy.signal as ss

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]


def find_root(base="/kaggle/input"):
    """Папка общего набора среди подключённых источников."""
    hits = glob.glob(f"{base}/**/h1_shared_meta.csv", recursive=True)
    assert hits, "общий набор не подключён: добавьте zakharsann/h1-prep-shared в kernel_sources"
    return os.path.dirname(hits[0])


def load_meta(root):
    """Таблица записей и описание набора."""
    meta = pd.read_csv(f"{root}/h1_shared_meta.csv")
    info = json.load(open(f"{root}/h1_shared_info.json", encoding="utf-8"))
    return meta, info


_SHARDS = {}
def _shard(root, fs, name):
    k = (root, fs, name)
    if k not in _SHARDS:
        _SHARDS[k] = np.load(f"{root}/fs{fs}/{name}", mmap_mode="r")
    return _SHARDS[k]


def read_record(root, row, fs=200):
    """Одна запись: (19, n) float16, мкВ. row — строка meta (Series или dict).
    Через mmap — для единичных записей; много записей читать через read_all (на /kaggle/input mmap очень медленный)."""
    n, off = int(row[f"n{fs}"]), int(row[f"off{fs}"])
    return np.array(_shard(root, fs, row[f"shard{fs}"])[off:off + 19 * n]).reshape(19, n)


def read_all(root, meta, fs=200, threads=4):
    """Все записи meta в словарь id -> (19, n) float16: каждый файл-кусок читается целиком последовательно, записи режутся в памяти."""
    def load(item):
        name, g = item
        arr = np.load(f"{root}/fs{fs}/{name}")
        return [(i, arr[o:o + 19 * n].reshape(19, n).copy()) for i, o, n in zip(g["id"], g[f"off{fs}"], g[f"n{fs}"])]
    out = {}
    with ThreadPoolExecutor(threads) as ex:
        for part in ex.map(load, list(meta.groupby(f"shard{fs}"))):
            out.update(part)
    return {i: out[i] for i in meta["id"]}


def windows(X, L, S=None):
    """Окна длиной L с шагом S (по умолчанию S = L, подряд): (k, 19, L)."""
    S = S or L
    k = (X.shape[-1] - L) // S + 1 if X.shape[-1] >= L else 0
    if k <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    return np.lib.stride_tricks.sliding_window_view(X, L, axis=-1)[..., ::S, :][..., :k, :].swapaxes(0, -2).copy()


def random_windows(X, L, K, rng):
    """K окон длиной L со случайным началом (любой отсчёт) — свежие окна на каждой эпохе: (K, 19, L)."""
    n = X.shape[-1] - L + 1
    if n <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    st = np.sort(rng.integers(0, n, K))
    return np.stack([X[..., s:s + L] for s in st])


def resample(X, fs_from, fs_to):
    """Пересчёт частоты (полифазный, как scipy.signal.resample_poly) по последней оси, float32 на выходе."""
    if fs_from == fs_to:
        return X
    fr = Fraction(int(fs_to), int(fs_from))
    return ss.resample_poly(np.asarray(X, np.float32), fr.numerator, fr.denominator, axis=-1).astype(np.float32)


def highpass(X, f, fs=200, order=4):
    """Дофильтровка сверху общей нижней границы 0,1 Гц (например, 0,3 Гц как у CBraMod): по всей записи, не по окну."""
    sos = ss.butter(order, f, btype="highpass", fs=fs, output="sos")
    return ss.sosfiltfilt(sos, np.asarray(X, np.float64), axis=-1).astype(np.float32)


In [ ]:
import os, glob, time, json, math
SMOKE = False   # True — короткая проверка на процессоре Kaggle (часть записей, 1 эпоха) + замер скорости на процессоре; её метрики не смотреть
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working"
T_NB = time.time()
END = T_NB + (75 if SMOKE else 100) * 60      # весь блокнот — не дольше 100 мин T4
left = lambda: (END - time.time()) / 60

SFREQ, SEG = 200, 10
L = SFREQ * SEG
K_TRAIN, N_VAL_EP, CAP_EVAL, HP = 8, 12, 60, 0.3
SEEDS = [0, 1, 2]

# модель и подготовка окна — один источник, он же сохраняется в выход как reve_v5_model.py
open(f"{OUT}/reve_v5_model.py", "w", encoding="utf-8").write(MODEL_SRC)
exec(compile(MODEL_SRC, "reve_v5_model.py", "exec"))
POS = positions(DEV)
print("позиции:", tuple(POS.shape))

MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]

def new_rv(seed):
    torch.manual_seed(seed)
    return RVNet(POS).to(DEV)

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

def predict_windows(model, Ws, bs=64):
    """Вероятность каждого окна: список массивов, по одному на запись."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = [torch.sigmoid(model(rv_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy() for b in range(0, len(W), bs)]
            out.append(np.concatenate(p) if p else np.zeros(0, np.float32))
    return out

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор", "| ядер процессора:", os.cpu_count())


## 2. Общий набор: чтение, ФВЧ 0,3 Гц, окна для выбора эпохи


In [ ]:
ROOT = find_root()
meta, info = load_meta(ROOT)
print("общий набор:", ROOT, {k: info.get(k) for k in ["fs", "nmt_scale_k", "nmt_train_cap_min", "size_gb"]})
need = meta[meta.split.isin(["fit", "val", "eval", "turkey"])].copy()
if SMOKE:
    pick = lambda s, n0, n1: pd.concat([need[(need.split == s) & (need.y == 0)].head(n0), need[(need.split == s) & (need.y == 1)].head(n1)])
    need = pd.concat([pick("fit", 24, 12), pick("val", 6, 4), pick("eval", 5, 5), pick("turkey", 5, 5)])
t0 = time.time()
REC = read_all(ROOT, need)
t_read = time.time() - t0

def hp16(i):
    """ФВЧ 0,3 Гц по записи целиком (как 0,3–75 Гц в REVE v2), хранение снова float16."""
    return i, highpass(REC[i], HP).astype(np.float16)
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, X in ex.map(hp16, list(REC)):
        REC[i] = X
print(f"прочитано {len(REC)} записей за {t_read:.0f} с, ФВЧ 0,3 Гц {time.time()-t0:.0f} с | {sum(x.nbytes for x in REC.values())/1e9:.1f} ГБ | осталось {left():.0f} мин")

fit, val, ev, tk = (need[need.split == s].reset_index(drop=True) for s in ["fit", "val", "eval", "turkey"])
print(pd.crosstab(need.split, need.y))
if not SMOKE:
    assert (len(val), len(ev), len(tk)) == (335, 185, 121), (len(val), len(ev), len(tk))
SETS = {"nmt_val": val, "nmt_eval": ev, "turkey": tk}
y_val = val.y.to_numpy()
VAL12 = [windows(REC[i], L)[spread(REC[i].shape[1] // L, N_VAL_EP)] for i in val.id]
print("окон подряд: " + ", ".join(f"{k} {sum(REC[i].shape[1] // L for i in d.id)}" for k, d in SETS.items()), "| для выбора эпохи:", sum(len(w) for w in VAL12))


## 3. Обучение: новые случайные окна + аугментации спектра, 3 запуска, веса в выход

In [ ]:
NOISE, BS, MAX_EP, PATIENCE, LR_HEAD, LR_BACK = 0.1, 32, (1 if SMOKE else 10), 2, 5e-4, 1e-5
def augment(x, g):
    B = x.shape[0]
    x = x + NOISE * torch.randn_like(x)
    m = torch.rand(B, device=x.device) < 0.5
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if g.random() < 0.5:
            x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

# --- новое в v5: аугментации до нормировки окна (сырые мкВ, на видеокарте) ---
P_TILT, TILT, P_DELTA, DELTA_G, P_CH, CH_G, PAD = 0.8, 0.5, 0.5, (0.7, 1.4), 0.5, (0.8, 1.25), SFREQ
FR = torch.fft.rfftfreq(L + 2 * PAD, 1 / SFREQ).to(DEV)
LOGF = torch.log(torch.clamp(FR, min=0.5) / 10.0)                                   # наклон относительно 10 Гц
_lo = torch.clamp((FR - 0.5) / 0.5, 0, 1); _hi = torch.clamp((4.5 - FR) / 0.5, 0, 1)
WDELTA = (0.5 - 0.5 * torch.cos(math.pi * _lo)) * (0.5 - 0.5 * torch.cos(math.pi * _hi))   # 1 в 1–4 Гц, плавные края 0,5 Гц

def spec_aug(x, g):
    """x: (B, 19, L) мкВ float32. Наклон спектра (f/10)^α, усиление 1–4 Гц, случайное усиление каналов."""
    B = x.shape[0]
    alpha = torch.tensor(g.uniform(-TILT, TILT, B) * (g.random(B) < P_TILT), dtype=torch.float32, device=x.device)
    dlog = torch.tensor(g.uniform(*np.log(DELTA_G), B) * (g.random(B) < P_DELTA), dtype=torch.float32, device=x.device)
    H = torch.exp(alpha[:, None] * LOGF[None] + dlog[:, None] * WDELTA[None])          # (B, частоты)
    xp = F.pad(x, (PAD, PAD), mode="reflect")
    x = torch.fft.irfft(torch.fft.rfft(xp, dim=-1) * H[:, None, :], n=L + 2 * PAD, dim=-1)[..., PAD:PAD + L]
    ch = np.exp(g.uniform(*np.log(CH_G), (B, 19))) ** (g.random((B, 1)) < P_CH)
    return x * torch.tensor(ch, dtype=torch.float32, device=x.device)[..., None]

def norm_t(x):
    """То же, что rv_input, на видеокарте: минус среднее канала, деление на общее СКО окна, обрезка ±15."""
    x = x - x.mean(-1, keepdim=True)
    return torch.clamp(x / (x.std(dim=(1, 2), keepdim=True, unbiased=False) + 1e-6), -15, 15)

# проверки: общий множитель нормировка убирает полностью; при α = 0 и без дельты/каналов окно не меняется
_W = np.random.default_rng(0).normal(0, 20, (2, 19, L)).astype(np.float32)
assert torch.allclose(rv_input(_W), rv_input(3.7 * _W), atol=1e-4)
assert torch.allclose(norm_t(torch.from_numpy(_W)), rv_input(_W), atol=1e-4)
_x = torch.from_numpy(_W).to(DEV)
_H = F.pad(_x, (PAD, PAD), mode="reflect")
assert torch.allclose(torch.fft.irfft(torch.fft.rfft(_H, dim=-1), n=L + 2 * PAD, dim=-1)[..., PAD:PAD + L], _x, atol=1e-3)
_g = np.random.default_rng(1); _a = spec_aug(_x.clone(), _g)
print("аугментации: проверки пройдены; пример — СКО окна до/после:", [round(float(v), 1) for v in _x.std(dim=(1, 2))], [round(float(v), 1) for v in _a.std(dim=(1, 2))])

def make_opt(model, total_steps):
    back = [p for n, p in model.named_parameters() if n.startswith("backbone")]
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK}, {"params": head, "lr": LR_HEAD}], weight_decay=0.05)
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t, params):
    with torch.autocast(**AMP):
        out = model(x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

# обучение: на каждой эпохе каждое окно — новое случайное окно своей записи
FITX = [REC[i] for i in fit.id]
y_fit = fit.y.to_numpy()
w_rec = np.where(y_fit == 1, 0.5 / (y_fit == 1).sum(), 0.5 / (y_fit == 0).sum()); w_rec /= w_rec.sum()
N_EP = len(FITX) * (1 if SMOKE else K_TRAIN)          # окон за эпоху (у v2 — 8 окон на запись, 15 103)
def fresh_batch(j, g):
    out = np.empty((len(j), 19, L), np.float16)
    for b, i in enumerate(j):
        s = int(g.integers(0, FITX[i].shape[1] - L + 1)); out[b] = FITX[i][:, s:s + L]
    return out
print(f"записей fit: {len(FITX)} (патология {y_fit.mean():.2f}), окон за эпоху: {N_EP}")

def train_run(seed, deadline):
    """До MAX_EP эпох; остановка, если отложенный ROC AUC не растёт PATIENCE эпох подряд или кончается время. Возвращается лучшая эпоха."""
    g = np.random.default_rng(seed)
    model = new_rv(seed)
    params = [p for p in model.parameters() if p.requires_grad]
    steps_ep = int(np.ceil(N_EP / BS))
    opt, sched = make_opt(model, MAX_EP * steps_ep)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad, stop = -1, None, 0, [], 0, "макс. эпох"
    t0 = time.time()
    for ep in range(MAX_EP):
        model.train(); losses = []
        recs = g.choice(len(FITX), size=N_EP, replace=True, p=w_rec)
        for b in range(0, N_EP, BS):
            j = recs[b:b + BS]
            xb = norm_t(spec_aug(torch.from_numpy(fresh_batch(j, g)).to(DEV).float(), g))
            l, bad = step(model, opt, sched, scaler, augment(xb, g), torch.tensor(y_fit[j], device=DEV), params)
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, [np.median(p) for p in predict_windows(model, VAL12)])
        hist.append((round(float(np.nanmean(losses)), 3), round(float(auc), 3)))
        print(f"  запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с | осталось {left():.0f} мин", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep + 1 - best_ep >= PATIENCE:
            stop = "плато"; break
        if time.time() + (time.time() - t0) / (ep + 1) > deadline:
            stop = "время"; break
    print(f"  запуск {seed}: остановка — {stop}, лучшая эпоха {best_ep}, {(time.time()-t0)/60:.1f} мин", flush=True)
    model.load_state_dict(best_state)
    return model, dict(best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad, stop=stop, n_ep=len(hist),
                       minutes=round((time.time() - t0) / 60, 1), min_per_ep=round((time.time() - t0) / 60 / len(hist), 2))

def predict_records(model, ids):
    """Все окна 10 с подряд с начала записи."""
    return predict_windows(model, (windows(REC[i], L)[:6] if SMOKE else windows(REC[i], L) for i in ids))

CFG_SAVE = dict(
    model="REVE v5 (Г1), дообучение на NMT train (fit), блокнот zakharsann/h1-reve-v5-aug",
    backbone="brain-bzh/reve-base", positions="brain-bzh/reve-positions", channels=EL, position_names=[NEW.get(e, e) for e in EL],
    head=dict(layers="Dropout(0.3) -> Linear(19*512, 256) -> GELU -> Dropout(0.3) -> Linear(256, 1)",
              input="выход основы (B, 19, T, 512) -> среднее по T -> flatten 19*512", in_features=19 * 512, hidden=256, dropout=0.3, out=1),
    state_dict="ключи backbone.* (REVE) и head.*; fp32; класс RVNet в reve_v5_model.py",
    input=dict(sfreq=SFREQ, window_s=SEG, window_samples=L, reference="средний по 19 каналам",
               filter="0,1–75 Гц Баттерворт 4 (sosfiltfilt) + режекция 50 Гц (iirnotch Q=30, filtfilt) + ФВЧ 0,3 Гц Баттерворт 4 (sosfiltfilt), по записи целиком",
               norm="из каждого канала вычесть его среднее по окну; окно поделить на общее СКО по всем каналам и отсчётам (+1e-6); обрезка ±15",
               unit="мкВ (после нормировки абсолютная амплитуда не важна)"),
    output="логит; вероятность патологии/эпилепсии = sigmoid(логит)",
    train=dict(data="NMT train, split=fit общего набора h1-prep-shared", lr_backbone=LR_BACK, lr_head=LR_HEAD, weight_decay=0.05, batch=BS,
               windows_per_epoch="8 x число записей fit, каждое окно — новое случайное", max_epochs=MAX_EP, patience=PATIENCE,
               augment="до нормировки: наклон спектра (f/10 Гц)^α, α~U(-0,5;0,5) p=0,8; 1–4 Гц ×U(0,7;1,4) p=0,5; усиление каналов log-U(0,8;1,25) p=0,5 (FFT, зеркальное дополнение 1 с); после нормировки: шум 0,1, зануление 1–2 каналов p=0,5, замена полушарий p=0,5, сдвиг до ±2 с"),
    record_score="оценка записи — медиана или среднее верхних 20 % вероятностей окон 10 с")

PRED_MIN, SPEED_MIN = (0.5 if SMOKE else 3.5), 0
RUNS, T_TRAIN = [], time.time()
for k, seed in enumerate(SEEDS):
    n_left = len(SEEDS) - k
    budget = (left() - n_left * PRED_MIN - SPEED_MIN - 1.5) / n_left
    model, inf = train_run(seed, time.time() + 60 * budget)
    torch.save({"state_dict": {k_: v.detach().cpu() for k_, v in model.state_dict().items()},
                "config": {**CFG_SAVE, "seed": seed, "best_epoch": inf["best_ep"], "val_auc_12win_median": round(float(inf["best_auc"]), 4)}},
               f"{OUT}/reve_v5_seed{seed}.pt")
    t0 = time.time()
    P = {ds: predict_records(model, d.id) for ds, d in SETS.items()}
    pm = (time.time() - t0) / 60; PRED_MIN = max(PRED_MIN, pm)
    RUNS.append(dict(seed=seed, **inf, P=P))
    a = {ds: roc_auc_score(d.y, [np.median(p[spread(len(p), CAP_EVAL)]) for p in P[ds]]) for ds, d in SETS.items()}
    print(f"запуск {seed}: эпоха {inf['best_ep']} ({inf['stop']}, всего {inf['n_ep']}) | медиана 60 окон: отложенные {a['nmt_val']:.3f}, NMT eval {a['nmt_eval']:.3f}, Турция {a['turkey']:.3f} | "
          f"предсказания {pm:.1f} мин | осталось {left():.0f} мин", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")


## 4. Итоги: ROC AUC, пороговые метрики (порог — отложенные 15 %), Спирмен с амплитудой, csv по людям и по окнам

In [ ]:
top20 = lambda p: np.sort(p)[::-1][:int(np.ceil(0.2 * len(p)))].mean()
AGGS = {"медиана": np.median, "верхние 20 %": top20}
VIEWS = {"60 окон": lambda p: p[spread(len(p), CAP_EVAL)], "все окна": lambda p: p}
LAB = {"nmt_val": "отложенные 15 %", "nmt_eval": "NMT eval", "turkey": "Турция"}
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}

# оценка записи: (окна, сборка) -> набор -> (запуски, записи)
S = {(v, a): {ds: np.array([[f(vf(p)) for p in r["P"][ds]] for r in RUNS]) for ds in SETS} for v, vf in VIEWS.items() for a, f in AGGS.items()}
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
srows, trows, parts = [], [], []
for (v, a), D in S.items():
    ens = {ds: np.mean([rank(s) for s in D[ds]], 0) for ds in SETS}
    prob = {ds: D[ds].mean(0) for ds in SETS}
    per_run = {ds: np.array([roc_auc_score(SETS[ds].y, s) for s in D[ds]]) for ds in SETS}
    srows.append({"окна": v, "сборка": a, **{f"{LAB[ds]}, ансамбль": fmt(SETS[ds].y.to_numpy(), ens[ds]) for ds in SETS},
                  **{f"{LAB[ds]}, запуски": f"{per_run[ds].mean():.3f} ± {sd(per_run[ds]):.3f} ({' / '.join(f'{x:.2f}' for x in per_run[ds])})" for ds in SETS}})
    fpr, tpr, thr = roc_curve(y_val, prob["nmt_val"])
    for tn_, t in {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}.items():
        for ds in SETS:
            y = SETS[ds].y.to_numpy(); mm = metr_ci(y, prob[ds], t)
            trows.append({"модель": "REVE v5", "окна": v, "сборка": a, "порог": tn_, "значение порога": round(t, 3), "набор": LAB[ds], "n": len(y),
                          "примечание": "порог: отложенные 15 % NMT train (среднее вероятностей 3 запусков)" + ("" if ds == "nmt_val" else "; перенесён без изменений"),
                          "ROC AUC": fmt(y, prob[ds]), "accuracy": mm["acc"], "сбаланс. accuracy": mm["bacc"], "precision": mm["prec"],
                          "recall (чувств.)": mm["rec"], "специфичность": mm["spec"]})
    for ds, d in SETS.items():
        t = pd.DataFrame({"dataset": ds, "id": d.id, "y": d.y, "std_uv": d.std_uv, "std_raw_uv": d.std_raw_uv, "dur_s": d.dur_s, "окна": v, "сборка": a})
        for r, s in zip(RUNS, D[ds]): t[f"reve5_seed{r['seed']}"] = s
        t["reve5_ens_rank"], t["reve5_ens_prob"] = ens[ds], prob[ds]
        parts.append(t)
summary = pd.DataFrame(srows); summary.to_csv(f"{OUT}/reve_v5_summary.csv", index=False)
print(summary.to_string(index=False))
thr_tab = pd.DataFrame(trows); thr_tab.to_csv(f"{OUT}/reve_v5_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание", "модель"]).to_string(index=False))
pp = pd.concat(parts); pp.to_csv(f"{OUT}/reve_v5_per_person.csv", index=False)
for (v, a) in S:
    d = pp[(pp.dataset == "turkey") & (pp["окна"] == v) & (pp["сборка"] == a)]
    print(f"Связь с амплитудой записи на Турции (Спирмен, {v}, {a}): {spearmanr(d.reve5_ens_rank, d.std_raw_uv).correlation:.2f}")

runs = pd.DataFrame([{**{k: v for k, v in r.items() if k != "P"},
                      **{f"{ds}_{v}_{a}": round(roc_auc_score(SETS[ds].y, S[(v, a)][ds][j]), 4) for (v, a) in S for ds in SETS}}
                     for j, r in enumerate(RUNS)])
runs.to_csv(f"{OUT}/reve_v5_runs.csv", index=False)
# главная проверка: Спирмен оценка записи – амплитуда записи (std_uv) внутри классов, 95% ДИ — бутстреп по людям
def spear_ci(a, b, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(a), len(a)); v.append(spearmanr(a[i], b[i]).correlation)
    return spearmanr(a, b).correlation, *np.nanpercentile(v, [2.5, 97.5])
sp_rows = []
for (v, a) in S:
    for ds in SETS:
        d = pp[(pp.dataset == ds) & (pp["окна"] == v) & (pp["сборка"] == a)]
        for yv, g_ in d.groupby("y"):
            rho, lo, hi = spear_ci(g_.reve5_ens_prob.to_numpy(), g_.std_uv.to_numpy())
            sp_rows.append({"модель": "REVE v5", "окна": v, "сборка": a, "набор": LAB[ds], "класс": int(yv), "n": len(g_),
                            "Спирмен": round(rho, 3), "ДИ": f"{lo:+.2f}…{hi:+.2f}"})
sp = pd.DataFrame(sp_rows); sp.to_csv(f"{OUT}/reve_v5_amp_spearman.csv", index=False)
print(sp[sp["сборка"] == "верхние 20 %"].to_string(index=False))
print(runs[["seed", "best_ep", "n_ep", "stop", "best_auc", "hist", "n_bad", "minutes", "min_per_ep"]].to_string(index=False))

# по окнам: все окна подряд; in60 — входит ли окно в 60 равномерных (основные цифры)
wparts = []
for ds, d in SETS.items():
    for r in RUNS:
        P = r["P"][ds]; n = [len(p) for p in P]
        in60 = np.concatenate([np.isin(np.arange(k), spread(k, CAP_EVAL)) for k in n])
        wparts.append(pd.DataFrame({"dataset": ds, "id": np.repeat(d.id.to_numpy(), n), "y": np.repeat(d.y.to_numpy(), n),
                                    "window": np.concatenate([np.arange(k) for k in n]), "in60": in60.astype(int), "seed": r["seed"], "p": np.concatenate(P).round(6)}))
wdf = pd.concat(wparts)
wdf.to_csv(f"{OUT}/reve_v5_window_scores.csv.gz", index=False, compression="gzip")
print(f"оценки окон: {len(wdf)} строк, {os.path.getsize(f'{OUT}/reve_v5_window_scores.csv.gz')/1e6:.1f} МБ | осталось {left():.0f} мин")

print("файлы:", sorted(os.listdir(OUT)))
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")


## 5. Сравнение с REVE v4 (локально по csv)

Считается скриптом `h1_eeg/src/reve_v5_compare.py` из `results/reve_v4_per_person.csv`, `reve_v5_per_person.csv`, `eegdino_finetune_per_person.csv` (лёгкий расчёт, ~1 мин на процессоре). Парная разность ROC AUC — бутстреп по людям (2000). Калибровка на k людях Турции — как в `ensemble_recall80.py`/`reve_v4_turkey_local_threshold_calibration.csv`: Юден по k людям, метрики на остальных, 500 повторов; v4 и v5 считаются на **одних и тех же** выборках (поэтому цифры v4 на 0,01 отличаются от отчёта v4, где выборки были другие). Ансамбль с EEG-DINO — среднее вероятностей (REVE ансамбль 3 запусков, верхние 20 %; EEG-DINO — среднее 3 запусков).

In [1]:
# запуск из h1_eeg/results
%run ../src/reve_v5_compare.py


1. ROC AUC, v5 − v4 (парный бутстреп по людям)
     окна       сборка           набор   n  REVE v4  REVE v5  v5 − v4        95% ДИ
 60 окон верхние 20 % отложенные 15 % 335    0.906    0.914    0.008 -0.007…+0.023
 60 окон верхние 20 %        NMT eval 185    0.917    0.931    0.014 -0.010…+0.041
 60 окон верхние 20 %          Турция 121    0.915    0.890   -0.025 -0.062…+0.010
 60 окон      медиана отложенные 15 % 335    0.890    0.897    0.007 -0.009…+0.024
 60 окон      медиана        NMT eval 185    0.882    0.890    0.008 -0.018…+0.033
 60 окон      медиана          Турция 121    0.891    0.892    0.001 -0.034…+0.037
все окна верхние 20 % отложенные 15 % 335    0.908    0.915    0.008 -0.007…+0.023
все окна верхние 20 %        NMT eval 185    0.917    0.931    0.014 -0.010…+0.041
все окна верхние 20 %          Турция 121    0.910    0.885   -0.026 -0.064…+0.012
все окна      медиана отложенные 15 % 335    0.888    0.897    0.009 -0.008…+0.026
все окна      медиана        NMT eval 1

![Спирмен оценка–амплитуда](../results/figures/reve_v5_amp_spearman.png)

![Оценка против амплитуды](../results/figures/reve_v5_amp_scatter.png)


## 6. Вывод Kaggle (без служебных предупреждений)

GPU-сеанс `zakharsann/h1-reve-v5-aug` (T4, 60,7 мин). Короткая проверка на процессоре — `zakharsann/h1-reve-v5-aug-smoke` (метрики не смотреть).

```
0.00s - to python to disable frozen modules.
0.00s - to python to disable frozen modules.
позиции: (19, 3)
устройство: Tesla T4 | ядер процессора: 4
общий набор: /kaggle/input/notebooks/zakharsann/h1-prep-shared {'fs': [200], 'nmt_scale_k': 100.0, 'nmt_train_cap_min': None, 'size_gb': 14.56}
прочитано 2535 записей за 41 с, ФВЧ 0,3 Гц 84 с | 14.6 ГБ | осталось 97 мин
y          0    1
split            
eval      95   90
fit     1618  276
turkey    71   50
val      286   49
окон подряд: nmt_val 23525, nmt_eval 13118, turkey 15505 | для выбора эпохи: 4001
аугментации: проверки пройдены; пример — СКО окна до/после: [20.0, 20.0] [20.3, 20.0]
записей fit: 1894 (патология 0.15), окон за эпоху: 15152
  запуск 0 | эпоха 1: потери 0.559 | отложенные NMT 0.886 | шагов с NaN/inf 0 | 208 с | осталось 94 мин
  запуск 0 | эпоха 2: потери 0.430 | отложенные NMT 0.892 | шагов с NaN/inf 0 | 418 с | осталось 90 мин
  запуск 0 | эпоха 3: потери 0.388 | отложенные NMT 0.886 | шагов с NaN/inf 0 | 628 с | осталось 87 мин
  запуск 0 | эпоха 4: потери 0.356 | отложенные NMT 0.880 | шагов с NaN/inf 0 | 837 с | осталось 83 мин
  запуск 0: остановка — плато, лучшая эпоха 2, 14.0 мин
запуск 0: эпоха 2 (плато, всего 4) | медиана 60 окон: отложенные 0.892, NMT eval 0.881, Турция 0.912 | предсказания 3.4 мин | осталось 80 мин
  запуск 1 | эпоха 1: потери 0.557 | отложенные NMT 0.863 | шагов с NaN/inf 0 | 210 с | осталось 76 мин
  запуск 1 | эпоха 2: потери 0.440 | отложенные NMT 0.880 | шагов с NaN/inf 0 | 420 с | осталось 73 мин
  запуск 1 | эпоха 3: потери 0.395 | отложенные NMT 0.894 | шагов с NaN/inf 0 | 630 с | осталось 69 мин
  запуск 1 | эпоха 4: потери 0.366 | отложенные NMT 0.866 | шагов с NaN/inf 0 | 840 с | осталось 66 мин
  запуск 1 | эпоха 5: потери 0.342 | отложенные NMT 0.881 | шагов с NaN/inf 0 | 1050 с | осталось 62 мин
  запуск 1: остановка — плато, лучшая эпоха 3, 17.5 мин
запуск 1: эпоха 3 (плато, всего 5) | медиана 60 окон: отложенные 0.894, NMT eval 0.901, Турция 0.868 | предсказания 3.4 мин | осталось 59 мин
  запуск 2 | эпоха 1: потери 0.552 | отложенные NMT 0.868 | шагов с NaN/inf 0 | 210 с | осталось 55 мин
  запуск 2 | эпоха 2: потери 0.440 | отложенные NMT 0.894 | шагов с NaN/inf 0 | 420 с | осталось 52 мин
  запуск 2 | эпоха 3: потери 0.386 | отложенные NMT 0.883 | шагов с NaN/inf 0 | 630 с | осталось 48 мин
  запуск 2 | эпоха 4: потери 0.371 | отложенные NMT 0.869 | шагов с NaN/inf 0 | 841 с | осталось 45 мин
  запуск 2: остановка — плато, лучшая эпоха 2, 14.0 мин
запуск 2: эпоха 2 (плато, всего 4) | медиана 60 окон: отложенные 0.891, NMT eval 0.867, Турция 0.882 | предсказания 3.4 мин | осталось 41 мин
всё обучение: 55.7 мин
    окна       сборка отложенные 15 %, ансамбль NMT eval, ансамбль  Турция, ансамбль           отложенные 15 %, запуски                  NMT eval, запуски                    Турция, запуски
 60 окон      медиана         0.897 (0.85-0.94)  0.890 (0.84-0.93) 0.892 (0.83-0.95) 0.892 ± 0.002 (0.89 / 0.89 / 0.89) 0.883 ± 0.017 (0.88 / 0.90 / 0.87) 0.888 ± 0.023 (0.91 / 0.87 / 0.88)
 60 окон верхние 20 %         0.914 (0.87-0.95)  0.931 (0.89-0.96) 0.890 (0.83-0.94) 0.911 ± 0.002 (0.91 / 0.91 / 0.91) 0.923 ± 0.015 (0.92 / 0.94 / 0.91) 0.888 ± 0.032 (0.91 / 0.85 / 0.90)
все окна      медиана         0.897 (0.85-0.94)  0.891 (0.85-0.93) 0.896 (0.83-0.95) 0.893 ± 0.001 (0.89 / 0.89 / 0.89) 0.884 ± 0.017 (0.88 / 0.90 / 0.87) 0.891 ± 0.021 (0.91 / 0.87 / 0.89)
все окна верхние 20 %         0.915 (0.87-0.95)  0.931 (0.89-0.96) 0.885 (0.82-0.94) 0.913 ± 0.002 (0.91 / 0.91 / 0.91) 0.923 ± 0.016 (0.93 / 0.94 / 0.91) 0.883 ± 0.034 (0.91 / 0.84 / 0.90)
    окна       сборка                 порог  значение порога           набор   n           ROC AUC         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
 60 окон      медиана                  Юден            0.588 отложенные 15 % 335 0.896 (0.85-0.94) 0.78 (0.74-0.82)  0.82 (0.77-0.87) 0.39 (0.30-0.48) 0.88 (0.78-0.96) 0.77 (0.72-0.81)
 60 окон      медиана                  Юден            0.588        NMT eval 185 0.891 (0.84-0.93) 0.81 (0.75-0.86)  0.81 (0.75-0.86) 0.80 (0.72-0.88) 0.80 (0.71-0.88) 0.81 (0.73-0.89)
 60 окон      медиана                  Юден            0.588          Турция 121 0.890 (0.82-0.95) 0.64 (0.55-0.72)  0.56 (0.52-0.61) 1.00 (1.00-1.00) 0.12 (0.04-0.22) 1.00 (1.00-1.00)
 60 окон      медиана чувствительность 80 %            0.600 отложенные 15 % 335 0.896 (0.85-0.94) 0.78 (0.73-0.82)  0.79 (0.73-0.85) 0.38 (0.29-0.47) 0.82 (0.70-0.92) 0.77 (0.72-0.82)
 60 окон      медиана чувствительность 80 %            0.600        NMT eval 185 0.891 (0.84-0.93) 0.80 (0.74-0.85)  0.80 (0.74-0.86) 0.80 (0.71-0.88) 0.79 (0.70-0.87) 0.81 (0.73-0.89)
 60 окон      медиана чувствительность 80 %            0.600          Турция 121 0.890 (0.82-0.95) 0.63 (0.55-0.71)  0.55 (0.51-0.60) 1.00 (1.00-1.00) 0.10 (0.02-0.19) 1.00 (1.00-1.00)
 60 окон верхние 20 %                  Юден            0.859 отложенные 15 % 335 0.915 (0.87-0.95) 0.88 (0.85-0.92)  0.84 (0.77-0.90) 0.58 (0.45-0.70) 0.78 (0.65-0.89) 0.90 (0.87-0.94)
 60 окон верхние 20 %                  Юден            0.859        NMT eval 185 0.930 (0.90-0.96) 0.83 (0.78-0.89)  0.83 (0.78-0.88) 0.93 (0.86-0.98) 0.71 (0.61-0.80) 0.95 (0.90-0.99)
 60 окон верхние 20 %                  Юден            0.859          Турция 121 0.899 (0.84-0.95) 0.61 (0.53-0.69)  0.53 (0.50-0.57) 1.00 (1.00-1.00) 0.06 (0.00-0.13) 1.00 (1.00-1.00)
 60 окон верхние 20 % чувствительность 80 %            0.831 отложенные 15 % 335 0.915 (0.87-0.95) 0.85 (0.81-0.89)  0.84 (0.77-0.89) 0.50 (0.39-0.62) 0.82 (0.70-0.92) 0.86 (0.82-0.90)
 60 окон верхние 20 % чувствительность 80 %            0.831        NMT eval 185 0.930 (0.90-0.96) 0.83 (0.77-0.88)  0.83 (0.77-0.88) 0.86 (0.79-0.93) 0.77 (0.67-0.85) 0.88 (0.82-0.95)
 60 окон верхние 20 % чувствительность 80 %            0.831          Турция 121 0.899 (0.84-0.95) 0.61 (0.53-0.69)  0.53 (0.50-0.57) 1.00 (1.00-1.00) 0.06 (0.00-0.13) 1.00 (1.00-1.00)
все окна      медиана                  Юден            0.569 отложенные 15 % 335 0.897 (0.85-0.94) 0.77 (0.72-0.81)  0.81 (0.76-0.86) 0.37 (0.28-0.46) 0.88 (0.78-0.96) 0.75 (0.70-0.79)
все окна      медиана                  Юден            0.569        NMT eval 185 0.891 (0.84-0.93) 0.81 (0.75-0.86)  0.81 (0.75-0.87) 0.80 (0.71-0.88) 0.82 (0.74-0.90) 0.80 (0.71-0.88)
все окна      медиана                  Юден            0.569          Турция 121 0.895 (0.83-0.95) 0.64 (0.55-0.72)  0.56 (0.52-0.61) 1.00 (1.00-1.00) 0.12 (0.04-0.22) 1.00 (1.00-1.00)
все окна      медиана чувствительность 80 %            0.600 отложенные 15 % 335 0.897 (0.85-0.94) 0.79 (0.74-0.83)  0.80 (0.74-0.85) 0.39 (0.30-0.48) 0.82 (0.70-0.91) 0.78 (0.73-0.83)
все окна      медиана чувствительность 80 %            0.600        NMT eval 185 0.891 (0.84-0.93) 0.79 (0.73-0.85)  0.79 (0.73-0.85) 0.79 (0.71-0.88) 0.77 (0.68-0.85) 0.81 (0.73-0.89)
все окна      медиана чувствительность 80 %            0.600          Турция 121 0.895 (0.83-0.95) 0.63 (0.55-0.71)  0.55 (0.51-0.60) 1.00 (1.00-1.00) 0.10 (0.02-0.19) 1.00 (1.00-1.00)
все окна верхние 20 %                  Юден            0.852 отложенные 15 % 335 0.916 (0.88-0.95) 0.88 (0.84-0.91)  0.84 (0.78-0.90) 0.56 (0.44-0.68) 0.80 (0.67-0.91) 0.89 (0.85-0.93)
все окна верхние 20 %                  Юден            0.852        NMT eval 185 0.931 (0.90-0.96) 0.83 (0.78-0.89)  0.83 (0.78-0.88) 0.93 (0.86-0.98) 0.71 (0.61-0.81) 0.95 (0.90-0.99)
все окна верхние 20 %                  Юден            0.852          Турция 121 0.892 (0.83-0.95) 0.61 (0.53-0.69)  0.53 (0.50-0.57) 1.00 (1.00-1.00) 0.06 (0.00-0.13) 1.00 (1.00-1.00)
все окна верхние 20 % чувствительность 80 %            0.831 отложенные 15 % 335 0.916 (0.88-0.95) 0.85 (0.81-0.89)  0.84 (0.77-0.89) 0.50 (0.39-0.62) 0.82 (0.70-0.92) 0.86 (0.82-0.90)
все окна верхние 20 % чувствительность 80 %            0.831        NMT eval 185 0.931 (0.90-0.96) 0.85 (0.80-0.90)  0.85 (0.80-0.90) 0.90 (0.83-0.96) 0.78 (0.69-0.86) 0.92 (0.85-0.97)
все окна верхние 20 % чувствительность 80 %            0.831          Турция 121 0.892 (0.83-0.95) 0.61 (0.53-0.69)  0.53 (0.50-0.57) 1.00 (1.00-1.00) 0.06 (0.00-0.13) 1.00 (1.00-1.00)
Связь с амплитудой записи на Турции (Спирмен, 60 окон, медиана): 0.51
Связь с амплитудой записи на Турции (Спирмен, 60 окон, верхние 20 %): 0.63
Связь с амплитудой записи на Турции (Спирмен, все окна, медиана): 0.50
Связь с амплитудой записи на Турции (Спирмен, все окна, верхние 20 %): 0.63
 модель     окна       сборка           набор  класс   n  Спирмен          ДИ
REVE v5  60 окон верхние 20 % отложенные 15 %      0 286    0.411 +0.31…+0.50
REVE v5  60 окон верхние 20 % отложенные 15 %      1  49    0.649 +0.38…+0.83
REVE v5  60 окон верхние 20 %        NMT eval      0  95    0.475 +0.31…+0.61
REVE v5  60 окон верхние 20 %        NMT eval      1  90    0.213 -0.01…+0.43
REVE v5  60 окон верхние 20 %          Турция      0  71    0.416 +0.17…+0.61
REVE v5  60 окон верхние 20 %          Турция      1  50    0.606 +0.40…+0.76
REVE v5 все окна верхние 20 % отложенные 15 %      0 286    0.413 +0.31…+0.51
REVE v5 все окна верхние 20 % отложенные 15 %      1  49    0.641 +0.36…+0.83
REVE v5 все окна верхние 20 %        NMT eval      0  95    0.473 +0.31…+0.61
REVE v5 все окна верхние 20 %        NMT eval      1  90    0.209 -0.01…+0.43
REVE v5 все окна верхние 20 %          Турция      0  71    0.423 +0.19…+0.62
REVE v5 все окна верхние 20 %          Турция      1  50    0.598 +0.40…+0.75
 seed  best_ep  n_ep  stop  best_auc                                                                           hist  n_bad  minutes  min_per_ep
    0        2     4 плато  0.892322                 [(0.559, 0.886), (0.43, 0.892), (0.388, 0.886), (0.356, 0.88)]      0     14.0        3.49
    1        3     5 плато  0.893892 [(0.557, 0.863), (0.44, 0.88), (0.395, 0.894), (0.366, 0.866), (0.342, 0.881)]      0     17.5        3.50
    2        2     4 плато  0.893892                [(0.552, 0.868), (0.44, 0.894), (0.386, 0.883), (0.371, 0.869)]      0     14.0        3.50
оценки окон: 156444 строк, 1.0 МБ | осталось 39 мин

всё время блокнота: 60.7 мин
```
